In [1]:
import os

# Set protobuf implementation before any other imports
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import glob
import math
import re

import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow import keras
from functools import partial
from sklearn.model_selection import train_test_split

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from PIL import Image

tf.config.optimizer.set_jit(True)
tf.config.threading.set_inter_op_parallelism_threads(4)
tf.config.threading.set_intra_op_parallelism_threads(4)

print("Tensorflow version " + tf.__version__)

if tf.config.list_physical_devices("GPU"):
    from tensorflow.keras import mixed_precision

    mixed_precision.set_global_policy("mixed_float16")

tf.random.set_seed(42)
np.random.seed(42)




2026-05-02 09:56:01.607683: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 09:56:01.632042: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777715761.666925      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777715761.675426      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 09:56:01.706367: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Tensorflow version 2.18.0


2026-05-02 09:56:20.008581: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
IMAGE_SIZE = 128  # keep original resolution
BATCH_SIZE = 128  # larger batch for better GPU utilization
NUM_CLASSES = 5
EPOCHS = 4

TRAIN_CSV = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
TRAIN_IMG_DIR = "/kaggle/input/cassava-leaf-disease-classification/train_images"
TEST_IMG_DIR = "/kaggle/input/cassava-leaf-disease-classification/test_images"

train_df = pd.read_csv(TRAIN_CSV)
train_df["label"] = train_df["label"].astype(int)

train_paths = (
    train_df["image_id"].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x)).values
)
train_labels = train_df["label"].astype(np.int32).values


def _load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = img / 255.0
    return img, label


train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = (
    train_ds.map(
        _load_and_preprocess,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=False,
    )
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg",
)

base_model.trainable = False

x = base_model.output
output = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(train_ds, epochs=EPOCHS, verbose=2)




9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/4
147/147 - 100s - 679ms/step - accuracy: 0.6562 - loss: 0.9753
Epoch 2/4
147/147 - 77s - 526ms/step - accuracy: 0.7086 - loss: 0.8000
Epoch 3/4
147/147 - 76s - 514ms/step - accuracy: 0.7244 - loss: 0.7565
Epoch 4/4
147/147 - 78s - 530ms/step - accuracy: 0.7330 - loss: 0.7328


In [3]:
def _load_image(path):
    """Load and preprocess a single image path (no label)."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = img / 255.0
    return img




In [4]:
def get_preds_model_list(image_dir, model_obj_list):
    """Ensemble prediction over a list of models using batched tf.data pipelines."""
    paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    model_preds = []
    for mod in model_obj_list:
        preds = mod.predict(ds, verbose=0)  # (N, NUM_CLASSES)
        model_preds.append(preds)
    avg_pred = np.mean(np.stack(model_preds, axis=0), axis=0)  # (N, NUM_CLASSES)

    labels = np.argmax(avg_pred, axis=1).astype(int)
    img_ids = [os.path.basename(p) for p in paths]
    return pd.DataFrame({"image_id": img_ids, "label": labels})




In [5]:
def get_preds(image_dir, model_obj):
    """Single‑model prediction helper using a batched pipeline."""
    paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    preds = model_obj.predict(ds, verbose=0)  # (N, NUM_CLASSES)
    labels = np.argmax(preds, axis=1).astype(int)
    img_ids = [os.path.basename(p) for p in paths]
    return pd.DataFrame({"image_id": img_ids, "label": labels})




In [6]:
mod_lst = [model]  # single‑model ensemble
predict_df = get_preds_model_list(TEST_IMG_DIR, mod_lst)




In [7]:
predict_df.to_csv("submission.csv", index=False)
display(predict_df.head())

,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,3
3,1234555380.jpg,3
4,1234571117.jpg,3
